# Identifier les textes évoquant le concept de république

In [ ]:
# TODO : Matthias : aviser sur les exclusions liées aux noms de lois
# TODO: léo : commencer à nettoyer les autres exclusions (pays, professions, etc.)


In [1]:
import pandas as pd
import re

# TODO: aviser si vire id_orateur et utiliser id_acteur partout
df = pd.read_csv(
    "../data/interim/data_cleaning_grouped.csv",
    low_memory=False,
    dtype={
        "ID_orateur": str
    },  # désormais géré avant (ajout PA et id_acteur) vire quand mettra au propre
)
df.shape

(425561, 52)

## INTRODUIRE PRÉ-TRAITEMENT TEXTE

Pour simplifier la vie et faciliter aussi possibles perf d'un futur modèle, virer les parenthèses et balises ici pour s'économiser pas mal de choses du côté des noms de groupes

In [ ]:
# TODO : remarque matthias -> aviser possibles soucis accents etc. ?

# nettoyage basique du texte
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes (nécessaire pour regex)
    texte = texte.replace("’", "'")
    return texte


df["texte_brut"] = df["texte"]  # garder une version brute du texte
df["texte"] = df["texte"].apply(nettoyer_texte)

# Si pas géré avant (devrait le faire dans le 2 sinon, mais possible perte avec nettoyage)
# Supprimer les lignes où "texte" est manquant
# pas déconnant de le garder là avec éventuelles suppressions dues au nettoyage
df = df.dropna(subset=["texte"])
df.shape

(425545, 53)

## Regex

Logique de la tentative :
- regex
- mais exclure certains termes
- mais comme les termes exclus peuvent apparaitre aussi avec les termes voulus, éviter de chainer et finir par virer des trucs qu'on aurait voulu (les idées républicaines sont menacées par Les Républicains)

In [ ]:
# préparer les pays à exclure
with open("../data/raw/liste_pays_republique_stable.txt", "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# créer un pattern regex pour les pays, plus bas on rendra le groupe non capturant
pattern_pays = r"|".join(re.escape(p) for p in liste_pays)


# Regex du champ lexical République (simplifié ici)

pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure
# logique : création de groupes (?:…) non capturant
# car utilisé juste pour les positions, pas besoin de les récupérer

# Expressions à exclure - casse exacte
# possible cas du féminin… mais pas d'occurrence dans la base avec nos exclusions
pattern_excl_case_sensitive = re.compile(
    r"(?:\b[LlDd]es Républicains\b)"  # garde la casse pour identifier le parti (et pas un adjectif)
    r"|(?:\baux Républicains\b)"  # idem majuscule pour le groupe
    r"|(?:\bsénateurs? Républicains?\b)"  # pas de féminin dans la base après exclu, mais aviser
    r"|(?:\bdéputés? Républicains?\b)"  # pas de féminin dans la base après exclu, mais aviser
)

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # partis et groupes politiques
    r"(?:\bgauche démocrate et républicaine\b)"  # premier sans |
    r"|(?:\brépublique en marche\b)"
    r"|(?:\bsocialiste, écologiste et républicain\b)"
    # TODO: reprendre liste exclusions
    # ex : Communiste, républicain, citoyen et écologiste
    # -> aviser avec et sans virgules selon nettoyage texte + feintes dans la graphie utilisée
    # fonctions et institutions
    r"|(?:\bprésident[s]? de la république\b)"
    r"|(?:\bprésidence[s]? de la république\b)"
    r"|(?:\bprocureur[s]? de la république\b)"
    r"|(?:\bcour[s]? de justice de la république\b)"
    r"|(?:\bcour[s]? de sûreté de la république\b)"
    r"|(?:\badministration générale de la république\b)"
    r"|(?:\bgouvernement de la république française\b)"
    # expression et législations
    r"|(?:\bcontrat d[’']engagement républicain\b)"  # à aviser
    # pays
    r"|(?:\brépublique[s]? soviétique[s]?\b)"  # pas un pays mais des expressions : aviser
    r"|(?:\brépublique[s]? de Weimar\b)"
    r"|(?:\b(?:" + pattern_pays + r")\b)",  # ajout des exclusions de pays
    re.I,
)


def contains_lexical_outside_excl(text):
    # TODO : aviser si veut utiliser spans triés et bisect pour optimiser la vérification des positions,
    # ou si pas besoin (en fonction du nombre d'exclusions et de la longueur des textes)
    # et on pourrait merger les positions de spans d'exclusions pour accélérer ?
    # mais pas indispensable ici et plus compliqué ?

    # si pas de match lexical inutile d'aller plus loin
    if not pattern_lexical.search(text):
        return False

    # Collecter les spans exclus
    # en ajoutant les exclusions sensibles et insensibles à la casse
    excl_positions = [m.span() for m in pattern_excl_case_sensitive.finditer(text)] + [
        m.span() for m in pattern_excl_case_insensitive.finditer(text)
    ]

    # Fonction pour vérifier si une position est dans une zone exclue
    # optimisable avec bisect si besoin
    def in_excl(pos):
        # return any(start <= pos < end for start, end in excl_positions) # equivalent mais moins clair
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurrences du champ lexical
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False


In [ ]:
# TODO : ajouter avancée matthias sur republicains
# TODO : doute sur ce qu'il veut (si bien imp de la maj ou si re.I)


En détail : 
« [Ll]es Républicains » 6686 ; « aux Républicains » (51); « [Dd]es Républicains » (428); « [Cc]ollègue[s]? Républicains » (26) « sénateurs Républicains »(2); « députés Républicains » (6); « entre Républicains » (4) « ex-Républicains » (1); « anciens Républicains » (1), « seuls Républicains » (1); « parlementaires Républicains » (1), « élus Républicains » (2); « groupeLes Républicains » (1); « Les Républicain » (4); « [Nn]ous Républicains » (2); « certains Républicains » (4); « élus Républicains » (2); « nos amis Républicains » (1); « droite, Républicains et macronistes » (1); « Républicains-Front national » (1); « Macronistes, Républicains, lepénistes »(1); « Rassemblement national, Républicains et macronistes » (1): (Attention, enlever sensitif à la casse) 



Ligne de code : 

r"\b[Ll]es Républicains|[Dd]es Républicains|aux Républicains|sénateurs Républicains|députés Républicains|entre Républicains|[Cc]ollègue[s]? Républicain[s]?|ex-Républicains|anciens Républicains|seuls Républicains|parlementaires Républicains|élus Républicains|groupeLes Républicains|Les Républicain|[Nn]ous? Républicains|certains Républicains|élus Républicains|nos amis Républicains|droite, Républicains et macronistes|Républicains-Front national|Républicains, lepénistes|Rassemblement national, Républicains\b",
    #re.I,

In [4]:
# # bloc d'essai
# mon_texte = "Les Députés Républicains ont voté une loi."  # devrait être exclu
# contains_lexical_outside_excl(mon_texte)


In [5]:
# Appliquer sur la colonne
df["repu_match_valide"] = df["texte"].apply(contains_lexical_outside_excl)

In [6]:
df["repu_match_valide"].value_counts()

repu_match_valide
False    414228
True      11317
Name: count, dtype: int64

In [7]:
# trace Matthias, aviser avec lui :)
# # Exporter fichier avec 2 colonnes pour calcul avec proportions
df.to_csv(
    "../data/interim/df_repu_proportion.csv",
    index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
)

In [8]:
df_match = df[df["repu_match_valide"]]
df_match.shape

(11317, 54)

In [9]:
df_match.to_csv(
    "../data/interim/df_repu.csv",
    index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
)

## Cf trace version matthias pour retrouver les noms de fichier
# # Exporter fichier uniquement avec Rep pour valeur absolues
# df_match.to_csv(
#     "../data/interim/df_regroup_repu_absolu.csv",
#     index=False,
#     # quoting=csv.QUOTE_ALL,  # not needed anymore ?
# )

In [10]:
# vérification écriture/lecture ok
print("df_match shape:", df_match.shape)

df_test = pd.read_csv("../data/interim/df_repu.csv", low_memory=False)

print("df_test shape (après export import): ", df_test.shape)

df_match shape: (11317, 54)
df_test shape (après export import):  (11317, 54)


In [11]:
# # Rajout en test d'une autre colonne avec le nombre de fois où la République apparait
# # À tester/voir si fonctionne bien mais en tout cas absence de cas avec false et au moins 1

# def count_lexical_outside_excl(text):
#     if pd.isna(text):
#         return 0

#     # Trouver les positions des expressions exclues
#     excl_positions = []
#     excl_positions.extend(
#         [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
#     )
#     excl_positions.extend(
#         [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
#     )

#     # Fonction pour vérifier si une position est dans une zone exclue
#     def in_excl(pos):
#         for start, end in excl_positions:
#             if start <= pos < end:
#                 return True
#         return False

#     # Compter les occurrences valides
#     count = 0
#     for match in pattern_lexical.finditer(text):
#         start_pos = match.start()
#         if not in_excl(start_pos):
#             count += 1

#     return count

In [12]:
# df["nombre_correspondances"] = df["texte"].apply(count_lexical_outside_excl)